# Notebook 05 — Pipeline Walkthrough
## Class 8 Skill: Turn analysis into a repeatable pipeline; include logging, checks, rerun behaviour, failure handling

This notebook traces the complete pipeline from raw inputs to metric outputs.
It also demonstrates:
- Idempotent rerun behaviour
- What a failure looks like and how it's handled
- Where the outputs are written
- The pipeline log structure

> FDE mindset: A pipeline that can't be re-run reliably is not a pipeline — it's a one-off script.

In [ ]:
import sys
import subprocess
from pathlib import Path
sys.path.insert(0, '../pipeline')

from config import METRICS_CSV, VALIDATION_CSV, PIPELINE_LOG, PROCESSED_TRIPS

print('Pipeline output paths:')
print(f'  Processed trips : {PROCESSED_TRIPS}')
print(f'  Metrics report  : {METRICS_CSV}')
print(f'  Validation log  : {VALIDATION_CSV}')
print(f'  Pipeline log    : {PIPELINE_LOG}')

## 1. Pipeline Architecture

```
  Raw Sources
  ─────────────────────────────────────────────────────────────────
  S1: TLC Parquet ──┐
  S2: Zone CSV ─────┤  INGEST (ingest.py)
  S3: SODA API ─────┤   • HTTP download (stream, 1 MB chunks)
  S4: Vendor CSV ───┘   • REST API call
                         • Local file read
                         • Completeness checks (row count, schema, nulls)
                              ↓
                    VALIDATE (validate.py)
                         • 8 business-oriented rules applied
                         • Multi-reason rejection (all failures logged)
                         • Rejection log → outputs/validation_log.csv
                              ↓
                    TRANSFORM (transform.py)
                         • Zone join (pickup + dropoff)
                         • Vendor join
                         • Derived columns (duration, speed, fare_per_mile)
                         • 5 metric aggregations
                              ↓
  Outputs
  ─────────────────────────────────────────────────────────────────
  data/processed/validated_trips.parquet
  outputs/metrics_report.csv
  outputs/validation_log.csv
  outputs/pipeline_run.log
```

## 2. Run the Pipeline (End-to-End)

In [ ]:
print('Running pipeline.py ...')
print('(This will skip downloads if raw files already exist)')
print('─' * 60)

# Run as subprocess so we capture the real output
result = subprocess.run(
    [sys.executable, 'pipeline.py'],
    cwd=str(Path('../pipeline').resolve()),
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:500])

print(f'Exit code: {result.returncode}')
if result.returncode == 0:
    print('Pipeline completed SUCCESSFULLY.')
else:
    print('Pipeline FAILED. See stderr/log above.')

## 3. Idempotency — What Happens on a Re-run?

Run the pipeline a second time to confirm it produces the same output without side effects.

In [ ]:
import hashlib

def file_hash(path):
    if not Path(path).exists():
        return 'FILE_NOT_FOUND'
    return hashlib.md5(Path(path).read_bytes()).hexdigest()

# Hash outputs before second run
hash_before = {
    'metrics': file_hash(METRICS_CSV),
    'validation': file_hash(VALIDATION_CSV),
}

# Re-run
result2 = subprocess.run(
    [sys.executable, 'pipeline.py'],
    cwd=str(Path('../pipeline').resolve()),
    capture_output=True,
    text=True
)

hash_after = {
    'metrics': file_hash(METRICS_CSV),
    'validation': file_hash(VALIDATION_CSV),
}

print('Idempotency check (MD5 hashes before vs. after second run):')
for key in hash_before:
    match = 'SAME' if hash_before[key] == hash_after[key] else 'DIFFERENT'
    print(f'  {key:12s}: {match} ({hash_after[key][:8]}...)')

## 4. Failure Handling — What Happens When Something Goes Wrong?

In [ ]:
# Demonstrate what a RuntimeError looks like — simulate a bad schema
import pandas as pd
from validate import validate
from ingest import verify_retrieval_completeness

print('Simulating a corrupted dataset (missing required columns) ...')
bad_df = pd.DataFrame({'col1': [1, 2, 3]})  # missing all expected columns

try:
    verify_retrieval_completeness(bad_df)
except RuntimeError as e:
    print(f'RuntimeError caught correctly: {e}')
    print('Pipeline would exit with code 1.')

## 5. Pipeline Log Inspection

In [ ]:
print('Last 30 lines of pipeline_run.log:')
print('─' * 60)
if Path(PIPELINE_LOG).exists():
    lines = Path(PIPELINE_LOG).read_text(encoding='utf-8').splitlines()
    for line in lines[-30:]:
        print(line)
else:
    print('Log file not found — run the pipeline first.')

## 6. Final Outputs

In [ ]:
import pandas as pd

if Path(METRICS_CSV).exists():
    metrics_df = pd.read_csv(METRICS_CSV)
    print(f'metrics_report.csv — {len(metrics_df)} rows')
    print(metrics_df.to_string(index=False))
else:
    print('Metrics report not found — run the pipeline first.')

## 7. Known / Unknown / Assumption / Limitation Table

| Category | Item |
|---|---|
| **Known** | TLC Parquet has duplicate trip IDs across vendors in some months |
| **Known** | Tip is systematically NULL for cash payments (payment_type = 2) |
| **Known** | LocationID 264, 265 = 'Unknown' zones — trips retained but flagged |
| **Unknown** | Whether a SQL database upstream holds more authoritative data |
| **Unknown** | Exact reason for zero-fare trips (comp ride vs. meter test vs. data error) |
| **Assumption** | Jan 2024 file is representative for demonstrating a monthly pipeline |
| **Assumption** | VendorID 1 = CMT, VendorID 2 = VeriFone (from TLC data dictionary) |
| **Limitation** | Only Yellow Taxi covered — Green Taxi and FHV need schema mapping |
| **Limitation** | Speed uses zone-level granularity, not GPS coordinates |
| **Limitation** | Pipeline runs locally; scheduling (cron/Airflow) is outside scope |

---

## 8. FDE Judgement Call — For the Demo

**Question asked:** *"Why didn't you just drop zero-fare trips?"*

**Answer:**  
Zero-fare trips could be: (a) complimentary rides given to customers, (b) meter tests, or (c) data entry errors. Silently dropping them would hide this ambiguity from the business. Instead, we classified them as `ZERO_FARE` and logged them separately. This means:
- The revenue KPI correctly excludes them
- The compliance team can query validation_log.csv to see how many there are
- If the business later decides zero-fare trips are a KPI in themselves, the data is preserved

The rule threshold (`fare_amount > 0`) is documented in `config.py`, not buried in a `df.dropna()` call.